In [1]:
import os
import torch
from model.datasets import ChunkedTextDataset
from datasets import load_dataset
from transformers import AutoTokenizer
from torchdata.stateful_dataloader import StatefulDataLoader
from config.settings import PreTrainingSettings
from torchtyping import TensorType


import torch
from torch.utils.data import IterableDataset, DataLoader
#from torch.utils.data.dataloader import DataLoaderStateDict

/Users/michalekb/Documents/GitHub_Personal/chefformer_repo/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/michalekb/Documents/GitHub_Personal/chefformer_repo/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Define the IterableDataset for C4
class C4IterableDataset(IterableDataset):
    def __init__(self, dataset_stream):
        super().__init__()
        self.dataset_stream = dataset_stream

    def __iter__(self):
        for idx, example in enumerate(self.dataset_stream):
            #print(f"Processing example {idx}: {example}")
            yield example


In [4]:
 # Define dataset and dataloader
# C4 English only dataset
c4 = load_dataset("allenai/c4", "en", streaming=True, split='train')
c4 = c4.shuffle(seed=42, buffer_size=5)

# Wrap the dataset in the custom IterableDataset
dataset = C4IterableDataset(c4)
dataloader = StatefulDataLoader(dataset, batch_size=1)

In [5]:
for epoch in range(0, 3):
    for step, x in enumerate(dataloader):
        print(f'Step: {step}; data example: {x}')

        if step == 10:
            checkpoint_saved = True
            break  # Exit the inner loop after step 5

    if checkpoint_saved:
        torch.save({'dataloader_state_dict': dataloader.state_dict()}, './checkpoint.pth')
        print("Saved checkpoint")
        break  # Exit the outer loop after saving the checkpoint

Step: 0; data example: {'text': ['Anyone having a concept but not the experience can utilize ClickFunnels to start or broaden their business. The agency provides strategic business processes or funnels that entrepreneurs can employ, to compensate for lack of experience.\nSimply put a sales funnel is an automated process that contributes to a sale. Funnels can be big, or little, and if they’re done properly the right mixture of traffic leads you to success. A great video to see that explains what a sales funnel is can be found right here.\nIn just four short years, ClickFunnels has grown to over 55,000 clients and can rely over 200 millionaires among them. Creator Russell Brunson has built the fastest growing non-venture endorsed software company on the planet. Brunson started his company at the age of 12 when he gathered direct mail flyers. Throughout college, his hobby developed into an obsession. He offered everything from supplements to novels. Founding ClickFunnels has been Russell

In [5]:
checkpoint = torch.load('./checkpoint.pth')
dataloader.load_state_dict(checkpoint['dataloader_state_dict'])

/var/folders/5c/w704b_sj5ln4y89hvfqn_x900000gn/T/ipykernel_50621/304533561.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('./checkpoint.pth')


In [6]:
break_all = False
for epoch in range(0, 3):
    for step, x in enumerate(dataloader):
        print(f'Step: {step}; data example: {x}')

        if step == 20:
            break_all = True
            break
    if break_all:
        break

Neither dataset nor iter(dataset) defines state_dict/load_state_dict so we are naively fast-forwarding your dataset by 11 steps. For more efficient resumes, please implement `state_dict` and `load_state_dict` in your IterableDataset and/or iterator.


Step: 0; data example: {'text': ['Preheat the oven to 200°C/400°F. Lightly dust a work surface with flour, roll out half the pastry to approx 30 x 40cm and use a cutter to stamp out 15 squares. Place into silicone moulds, prick the bases with a fork. Repeat with the other half of the pastry. Brush with the beaten egg and sprinkle with parmesan. Bake for 8-10 minutes until golden. Remove from the oven and transfer to a wire rack to cool.\nWhen all the pastry cases are baked and cooled, place 1 tsp of hummus in each, top with a cherry tomato and a few chopped chives.'], 'timestamp': ['2019-04-21 00:59:32'], 'url': ['https://www.blog.lakeland.co.uk/recipe/hummus-cherry-tomato-mini-tartlets/']}
Step: 1; data example: {'text': ['Step 1: Obtain a CFD of the MR Jet with a good PISA formation under zoomed view. The ME 3CV, ME 4CV, ME 2CV, or ME AVLAX are good views. Use the view with the largest and high quality PISA formation.'], 'timestamp': ['2019-04-23 22:09:44'], 'url': ['https://e-echoca

In [2]:
# Set up tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load the full validation split
c4_val_full = load_dataset("allenai/c4", "en", streaming=True, split='validation')

In [5]:
# Convert the streaming dataset to a list of indices and shuffle deterministically
sampled_validation_size = 100  # Desired validation size
c4_val_sampled = c4_val_full.take(sampled_validation_size)


In [6]:
c4_val = ChunkedTextDataset(c4_val_sampled, tokenizer=tokenizer, chunk_size=512, overlap_size=50, shuffle=False, shuffle_buffer_size=10000)

In [7]:
# Set up dataloader for training
dataloader_val = StatefulDataLoader(c4_val, 
                                batch_size=PreTrainingSettings.batch_size, 
                                #collate_fn=batch_collator #,
                                #snapshot_every_n_steps=PreTrainingSettings.save_checkpoint_every
                                )

In [8]:
for cur_training_step, (input_ids, attention_mask) in enumerate(dataloader_val):
    print(input_ids)
    if cur_training_step == 3:
        break

tensor([[  464,  2415,   508,  ..., 17707, 49948,   284],
        [ 9826,   262,   976,  ...,   340,  1244,   307],
        [  353,    14,    83,  ...,   339,  5839,   503],
        ...,
        [12108,   286,  5828,  ...,    11,   290, 34205],
        [  783,  1498,   284,  ...,  1903,   355,   642],
        [11333,  3812,   257,  ..., 10019,   284, 15872]])
tensor([[  262,   983,   355,  ...,   468,   587,  3032],
        [ 2266,  2613,   318,  ...,   287,   995,  1165],
        [  351,   257,  5339,  ...,   710,    11,   314],
        ...,
        [ 1268,   376, 11211,  ...,   290,  2121,  7987],
        [  345,  1716,   257,  ...,  1016,   284,  1730],
        [  815,   307,  7981,  ...,   290, 39107,   477]])
tensor([[ 1551,   339,  1839,  ...,   355,   345,  6537],
        [  880,   852,    13,  ...,    13,   921,   460],
        [ 2728,  2089, 17644,  ...,  1767, 13502,    13],
        ...,
        [ 1064, 30438,   284,  ...,    11,  2685, 32141],
        [10693,    11,  9456,  